# Tarea 2 – Parte 1: Corpus y extracción
- **Fecha de extracción:** 2026-10-26
- **Fuentes y licencias:** detalladas en `fuentes.csv`
- **OCR:** PaddleOCR (versión impresa abajo), lang=es
- **ASR:** faster-whisper `base` + VAD Silero (vad_filter=True); diarización con pyannote si hay token HF

In [1]:
import os, glob, hashlib, re
import pandas as pd
import plotly.express as px
import paddleocr, faster_whisper
print("PaddleOCR:", paddleocr.__version__, "| faster-whisper:", faster_whisper.__version__)


FUENTES = {
    "impresos": "El Peruano",
    "fotos": "propias",
    "manuscritos": "apuntes",
    "audio":       "canal del MINCETUR, MIDIS Y MINDEF",
}

def sha256(p):
    with open(p, "rb") as f:
        return hashlib.sha256(f.read()).hexdigest()

docs = pd.DataFrame([{"archivo": p, "tipo": os.path.basename(os.path.dirname(p))}
                     for p in glob.glob("data/docs/*/*") if p.lower().endswith((".jpg", ".jpeg", ".png"))])
audios = pd.DataFrame([{"archivo_audio": p} for p in glob.glob("data/audio/*")
                       if p.lower().endswith((".mp3", ".wav", ".m4a", ".ogg", ".flac"))])

# Limpieza: duplicados por hash
for nombre, df, col in [("documentos", docs, "archivo"), ("audios", audios, "archivo_audio")]:
    if len(df):
        df["hash"] = df[col].apply(sha256)
        dup = df[df.duplicated("hash")]
        print(f"{nombre}: {len(df)} archivos, duplicados eliminados: {len(dup)} {dup[col].tolist()}")
docs = docs.drop_duplicates("hash").reset_index(drop=True) if len(docs) else docs
audios = audios.drop_duplicates("hash").reset_index(drop=True) if len(audios) else audios
print(docs["tipo"].value_counts() if len(docs) else "Sin documentos")

c:\TAREA 2\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PaddleOCR: 3.7.0 | faster-whisper: 1.2.1
documentos: 25 archivos, duplicados eliminados: 0 []
audios: 20 archivos, duplicados eliminados: 0 []
tipo
manuscritos    9
fotos          8
impresos       8
Name: count, dtype: int64


In [2]:
from paddleocr import PaddleOCR
PARAMS = dict(text_detection_model_name="PP-OCRv5_mobile_det",
              text_recognition_model_name="latin_PP-OCRv5_mobile_rec",
              use_doc_orientation_classify=False, use_doc_unwarping=False,
              use_textline_orientation=False, enable_mkldnn=False,
              text_det_limit_type="max", text_det_limit_side_len=960)
print("Parámetros OCR:", PARAMS)
ocr = PaddleOCR(**PARAMS)

textos, descartes = [], []
for p in docs["archivo"]:
    try:
        res = ocr.predict(p)[0]
        res.save_to_json("outputs/ocr_raw/")   # respaldo crudo
        res.save_to_img("outputs/ocr_raw/")    # imagen con recortes/cajas detectadas
        t = " ".join(res["rec_texts"])
    except Exception as e:
        t = ""; print("Error en", p, e)
    textos.append(t)
    if len(t.strip()) < 10:
        descartes.append((p, "ilegible: OCR casi vacío"))
docs["texto_ocr"] = textos
docs["modelo_ocr"] = f"PaddleOCR {paddleocr.__version__}"
print("Descartados por ilegibles:", descartes)
docs = docs[~docs["archivo"].isin([d[0] for d in descartes])].reset_index(drop=True)


Parámetros OCR: {'text_detection_model_name': 'PP-OCRv5_mobile_det', 'text_recognition_model_name': 'latin_PP-OCRv5_mobile_rec', 'use_doc_orientation_classify': False, 'use_doc_unwarping': False, 'use_textline_orientation': False, 'enable_mkldnn': False, 'text_det_limit_type': 'max', 'text_det_limit_side_len': 960}


c:\TAREA 2\.venv\Lib\site-packages\paddle\utils\cpp_extension\extension_utils.py:712: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
Creating model: ('PP-OCRv5_mobile_det', None, None)
Checking connectivity to the model hosters, this may take a while. To bypass this check, set `PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK` to `True`.
Using official model (PP-OCRv5_mobile_det), the model files will be automatically downloaded and saved in `C:\Users\LENOVO\.paddlex\official_models\PP-OCRv5_mobile_det`.
Fetching 6 files: 100%|██████████| 6/6 [00:06<00:00,  1.01s/it]
Creating model: ('latin_PP-OCRv5_mobile_rec', None, None)
Using official model (latin_PP-OCRv5_mobile_rec), the model files will be automatically downloaded and saved in `C:\Users\LENOVO\.paddlex\official_models\latin_PP-OCRv5_mobile_rec`.
Fetching 6 f

Descartados por ilegibles: []


In [ ]:
# Compatibilidad: la versión nueva de PyAV ya no acepta 'metadata_errors'
import av
_av_open = av.open
def _av_open_compat(*args, **kwargs):
    kwargs.pop("metadata_errors", None)
    return _av_open(*args, **kwargs)
av.open = _av_open_compat
from faster_whisper import WhisperModel
asr = WhisperModel("tiny", device="cpu", compute_type="int8")

HF_TOKEN = ""   # ✏️ si tienes token de Hugging Face, pégalo aquí; si no, déjalo vacío
diar = None
if HF_TOKEN:
    try:
        from pyannote.audio import Pipeline
        diar = Pipeline.from_pretrained("pyannote/speaker-diarization-3.1", token=HF_TOKEN)
    except Exception as e:
        print("Diarización no disponible:", e)

def t_srt(t):
    return f"{int(t//3600):02}:{int(t%3600//60):02}:{int(t%60):02},{int((t%1)*1000):03}"

filas = []
for i, p in enumerate(audios["archivo_audio"]):
    try:
        segs, info = asr.transcribe(p, language="es", vad_filter=True)
        segs = list(segs)
    except Exception as e:
        print("Error en", p, e); continue
    turnos = []
    if diar:
        try:
            turnos = [(t.start, t.end, s) for t, _, s in diar(p).itertracks(yield_label=True)]
        except Exception as e:
            print("Diarización falló en", p, e)
    def spk(s):
        sol = {}
        for a, b, h in turnos:
            sol[h] = sol.get(h, 0) + max(0, min(b, s.end) - max(a, s.start))
        return max(sol, key=sol.get) if sol else "SPK_00"
    base = "outputs/asr/" + os.path.splitext(os.path.basename(p))[0]
    with open(base + ".srt", "w", encoding="utf-8") as f:
        for k, s in enumerate(segs, 1):
            f.write(f"{k}\n{t_srt(s.start)} --> {t_srt(s.end)}\n[{spk(s)}] {s.text.strip()}\n\n")
    texto = " ".join(s.text.strip() for s in segs)
    with open(base + ".txt", "w", encoding="utf-8") as f:
        f.write(texto)
    if not texto.strip():
        print("Descartado (silencio):", p); continue
    filas.append({"id": f"A{i+1:03}", "archivo_audio": os.path.basename(p), "duracion_seg": round(info.duration, 1),
                  "idioma": info.language, "texto_asr": texto, "modelo_asr": "faster-whisper base + VAD"
                  + (" + pyannote 3.1" if diar else ""), "n_hablantes": len({t[2] for t in turnos}) if turnos else None,
                  "texto_referencia": ""})
trans = pd.DataFrame(filas, columns=["id","archivo_audio","duracion_seg","idioma","texto_asr","modelo_asr","n_hablantes","texto_referencia"])
print(len(trans), "clips |", round(trans["duracion_seg"].sum()/60, 1) if len(trans) else 0, "minutos")